In [1]:
import numpy as np 
import pandas as pd

In [2]:
df = pd.read_csv("data\im2latex_train.csv")



<>:1: SyntaxWarning: invalid escape sequence '\i'
<>:1: SyntaxWarning: invalid escape sequence '\i'
C:\Users\DANISH\AppData\Local\Temp\ipykernel_14308\3911428846.py:1: SyntaxWarning: invalid escape sequence '\i'
  df = pd.read_csv("data\im2latex_train.csv")


In [3]:
print(df.head())
print(df.shape)
print(df.columns.to_list())

                                             formula           image
0  \widetilde \gamma _ { \mathrm { h o p f } } \s...  66667cee5b.png
1  ( { \cal L } _ { a } g ) _ { i j } = 0 , \ \ \...  1cbb05a562.png
2  S _ { s t a t } = 2 \pi \sqrt { N _ { 5 } ^ { ...  ed164cc822.png
3  \hat { N } _ { 3 } = \sum \sp f _ { j = 1 } a ...  e265f9dc6b.png
4  \, ^ { * } d \, ^ { * } H = \kappa \, ^ { * } ...  242a58bc3a.png
(75275, 2)
['formula', 'image']


In [4]:
class Vocabulary:
    PAD = "<PAD>"   # index 0 — fills empty space in batches
    SOS = "<SOS>"   # index 1 — start of sequence, fed to decoder first
    EOS = "<EOS>"   # index 2 — model must learn to predict this to stop
    UNK = "<UNK>"   # index 3 — any token not seen during training

    def __init__(self):
        self.token2idx = {
            self.PAD: 0,
            self.SOS: 1,
            self.EOS: 2,
            self.UNK: 3,
        }
        self.idx2token = {v: k for k, v in self.token2idx.items()}

    def build(self, formulas: list[str], min_freq: int = 1):
        # Count every token across all formulas
        from collections import Counter
        counts = Counter()
        for formula in formulas:
            counts.update(formula.split())  # tokens are space-separated

        # Only keep tokens that appear at least min_freq times
        # Rare tokens just add noise — UNK handles them
        for token, freq in counts.items():
            if freq >= min_freq:
                idx = len(self.token2idx)
                self.token2idx[token] = idx
                self.idx2token[idx] = token

        print(f"Vocabulary size: {len(self)}")

    def encode(self, formula: str) -> list[int]:
        tokens = formula.split()
        return (
            [self.token2idx[self.SOS]]
            + [self.token2idx.get(t, self.token2idx[self.UNK]) for t in tokens]
            + [self.token2idx[self.EOS]]
        )

    def decode(self, indices: list[int]) -> str:
        tokens = []
        for idx in indices:
            token = self.idx2token.get(idx, self.UNK)
            if token == self.EOS:
                break
            if token not in (self.SOS, self.PAD):
                tokens.append(token)
        return " ".join(tokens)

    def __len__(self):
        return len(self.token2idx)

In [5]:
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

In [6]:


class Im2LatexDataset(Dataset):
    def __init__(
        self,
        csv_path: str,
        image_dir: str,
        vocab: Vocabulary,
        max_seq_len: int = 150,
        max_samples: int = None,
        image_height: int = 64,
        image_width: int = 400,
    ):
        self.image_dir = Path(image_dir)
        self.vocab = vocab
        self.max_seq_len = max_seq_len

        # Pandas reads the CSV — handles quoting automatically
        self.df = pd.read_csv(csv_path)

        if max_samples is not None:
            self.df = self.df.head(max_samples)

        # Drop rows where the image file doesn't exist on disk
        # numpy's vectorized operation — much faster than a Python loop
        exists = np.array([
            (self.image_dir / fname).exists()
            for fname in self.df["image"]
        ])
        self.df = self.df[exists].reset_index(drop=True)
        print(f"Loaded {len(self.df)} samples (dropped {(~exists).sum()} missing images)")

        # Image preprocessing pipeline
        # Every image gets resized to the same shape, converted to grayscale,
        # then normalized so pixel values are in [-1, 1] instead of [0, 255]
        self.transform = transforms.Compose([
            transforms.Resize((image_height, image_width)),
            transforms.Grayscale(num_output_channels=1),
            transforms.ToTensor(),              # HxWxC numpy → CxHxW tensor, scales to [0,1]
            transforms.Normalize((0.5,), (0.5,))  # shifts to [-1, 1]
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]   # pandas: get row by integer position

        # --- Image ---
        img_path = self.image_dir / row["image"]
        image = Image.open(img_path).convert("RGB")
        image = self.transform(image)   # returns a torch.Tensor of shape (1, H, W)

        # --- Formula ---
        encoded = self.vocab.encode(row["formula"])

        # Truncate if too long
        encoded = encoded[:self.max_seq_len]

        # Pad to max_seq_len with zeros (PAD index = 0)
        # numpy is convenient here: create a zero array then fill it
        padded = np.zeros(self.max_seq_len, dtype=np.int64)
        padded[:len(encoded)] = encoded

        formula = torch.tensor(padded, dtype=torch.long)

        return image, formula

In [7]:
def collate_fn(batch):
    # batch is a list of (image, formula) tuples — one per sample
    # DataLoader calls this to combine them into a single batch tensor
    images, formulas = zip(*batch)

    # Stack images: list of (1,H,W) tensors → one (B,1,H,W) tensor
    images = torch.stack(images, dim=0)

    # Stack formulas: list of (max_seq_len,) tensors → (B, max_seq_len)
    formulas = torch.stack(formulas, dim=0)

    return images, formulas

In [8]:
def build_dataset(data_dir: str):
    data_dir = Path(data_dir)

    # Load all formulas to build vocabulary
    formulas_path = data_dir / "im2latex_formulas.norm.csv"
    all_formulas = pd.read_csv(formulas_path, header=None, skiprows=1)[0].tolist()
    # skiprows=1 skips the "formulas" header line you saw in your data

    vocab = Vocabulary()
    vocab.build(all_formulas)

    train_dataset = Im2LatexDataset(
        csv_path=data_dir / "im2latex_train.csv",
        image_dir=data_dir / "formula_images_processed",
        vocab=vocab,
        max_samples = 10,
    )

    val_dataset = Im2LatexDataset(
        csv_path=data_dir / "im2latex_validate.csv",
        image_dir=data_dir / "formula_images_processed",
        vocab=vocab,
        max_samples=1,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=32,
        shuffle=True,        # shuffle training data every epoch
        num_workers=0,       # load data in parallel using 4 CPU threads
        collate_fn=collate_fn,
        pin_memory=True,     # speeds up CPU→GPU transfer
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=32,
        shuffle=False,       # never shuffle validation — results must be reproducible
        num_workers=0,
        collate_fn=collate_fn,
        pin_memory=True,
    )

    return train_loader, val_loader, vocab


if __name__ == "__main__":
    train_loader, val_loader, vocab = build_dataset("data")

    # Sanity check — grab one batch and inspect shapes
    images, formulas = next(iter(train_loader))
    print("Image batch shape:", images.shape)     # (32, 1, 64, 400)
    print("Formula batch shape:", formulas.shape) # (32, 150)
    print("Vocab size:", len(vocab))
    print("Sample formula decoded:", vocab.decode(formulas[0].tolist()))

Vocabulary size: 583
Loaded 10 samples (dropped 0 missing images)
Loaded 1 samples (dropped 0 missing images)
Image batch shape: torch.Size([10, 1, 64, 400])
Formula batch shape: torch.Size([10, 150])
Vocab size: 583
Sample formula decoded: \, ^ { * } d \, ^ { * } H = \kappa \, ^ { * } d \phi = J _ { B } .


c:\Users\DANISH\Desktop\CNN latex\venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [9]:
import torch
import torch.nn as nn
import torchvision.models as models

In [10]:
class CNNEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.cnn = nn.Sequential(
            # Block 1
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),         # (B, 64, 32, 200)

            # Block 2
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),         # (B, 128, 16, 100)

            # Block 3
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),         # (B, 256, 8, 50)

            # Block 4
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
        )                               # (B, 512, 8, 50)

    def forward(self, images):
        # images: (B, 1, 64, 400)
        x = self.cnn(images)            # (B, 512, 8, 50)

        B, C, H, W = x.shape
        x = x.view(B, C, H * W)        # (B, 512, 400)
        x = x.permute(0, 2, 1)         # (B, 400, 512)

        return x

In [11]:
class Attention(nn.Module):
    def __init__(self, encoder_dim, decoder_dim, attention_dim):
        super().__init__()

        # These three linear layers compute attention scores
        self.encoder_att = nn.Linear(encoder_dim, attention_dim)
        self.decoder_att = nn.Linear(decoder_dim, attention_dim)
        self.full_att    = nn.Linear(attention_dim, 1)
        self.relu        = nn.ReLU()
        self.softmax     = nn.Softmax(dim=1)

    def forward(self, encoder_out, decoder_hidden):
        # encoder_out:    (B, 64, 512)  — all spatial features from CNN
        # decoder_hidden: (B, 512)      — what the decoder knows so far

        att1 = self.encoder_att(encoder_out)      # (B, 64, attention_dim)
        att2 = self.decoder_att(decoder_hidden)   # (B, attention_dim)

        # att2 is (B, attention_dim) — we need to broadcast it across 64 positions
        # unsqueeze(1) makes it (B, 1, attention_dim) so addition works
        att  = self.full_att(self.relu(att1 + att2.unsqueeze(1))).squeeze(2)
        # att: (B, 64) — one raw score per spatial position

        alpha = self.softmax(att)                 # (B, 64) — scores sum to 1

        # Weighted sum of encoder features
        # alpha.unsqueeze(2): (B, 64, 1) — weight for each position
        # encoder_out:        (B, 64, 512)
        # multiply + sum across 64 positions -> context vector (B, 512)
        context = (encoder_out * alpha.unsqueeze(2)).sum(dim=1)

        return context, alpha

In [12]:
class LSTMDecoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, decoder_dim, encoder_dim, attention_dim, dropout=0.5):
        super().__init__()

        self.attention   = Attention(encoder_dim, decoder_dim, attention_dim)
        self.embedding   = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm_cell   = nn.LSTMCell(embed_dim + encoder_dim, decoder_dim)
        self.fc          = nn.Linear(decoder_dim, vocab_size)
        self.dropout     = nn.Dropout(dropout)

        # Initialize LSTM hidden state from encoder output
        self.init_h = nn.Linear(encoder_dim, decoder_dim)
        self.init_c = nn.Linear(encoder_dim, decoder_dim)

    def init_hidden(self, encoder_out):
        # encoder_out: (B, 64, 512)
        # Mean over the 64 spatial positions -> (B, 512)
        mean_enc = encoder_out.mean(dim=1)
        h = torch.tanh(self.init_h(mean_enc))   # (B, decoder_dim)
        c = torch.tanh(self.init_c(mean_enc))   # (B, decoder_dim)
        return h, c

    def forward(self, encoder_out, formulas):
        # encoder_out: (B, 64, 512)
        # formulas:    (B, max_seq_len) — token indices

        B          = encoder_out.size(0)
        max_len    = formulas.size(1) - 1   # -1 because last token has nothing to predict after it
        vocab_size = self.fc.out_features

        # Embed every token at once: (B, max_seq_len, embed_dim)
        embeddings = self.dropout(self.embedding(formulas))

        # Initialize hidden state from encoder
        h, c = self.init_hidden(encoder_out)

        # Store predictions
        predictions = torch.zeros(B, max_len, vocab_size)

        for t in range(max_len):
            # Get context vector from attention
            context, alpha = self.attention(encoder_out, h)

            # Input to LSTM: current token embedding + context vector
            # formulas[:, t] is the token at position t for every sample in the batch
            lstm_input = torch.cat([embeddings[:, t, :], context], dim=1)
            # (B, embed_dim + encoder_dim)

            # LSTMCell takes one step
            # h, c are the hidden and cell states — the LSTM's memory
            h, c = self.lstm_cell(lstm_input, (h, c))

            # Predict next token from hidden state
            pred = self.fc(self.dropout(h))     # (B, vocab_size)
            predictions[:, t, :] = pred

        return predictions

In [13]:
class Im2LatexModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, decoder_dim=512,
                 encoder_dim=512, attention_dim=256, dropout=0.5):
        super().__init__()
        self.encoder = CNNEncoder()
        self.decoder = LSTMDecoder(
            vocab_size=vocab_size,
            embed_dim=embed_dim,
            decoder_dim=decoder_dim,
            encoder_dim=encoder_dim,
            attention_dim=attention_dim,
            dropout=dropout,
        )

    def forward(self, images, formulas):
        encoder_out = self.encoder(images)           # (B, 64, 512)
        predictions = self.decoder(encoder_out, formulas)  # (B, max_len, vocab_size)
        return predictions

In [14]:
if __name__ == "__main__":
    vocab_size = 583
    model = Im2LatexModel(vocab_size=vocab_size)

    images   = torch.randn(4, 1, 64, 400)
    formulas = torch.randint(0, vocab_size, (4, 150))

    output = model(images, formulas)
    print("Output shape:", output.shape)      # (4, 149, 583)
    print("Model parameters:", sum(p.numel() for p in model.parameters()))

Output shape: torch.Size([4, 149, 583])
Model parameters: 5413832


In [15]:
import torch.optim as optim
from tqdm.notebook import tqdm

def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for images, formulas in tqdm(dataloader, desc="Training", leave=False):
        images = images.to(device)
        formulas = formulas.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        output = model(images, formulas)
        
        # Reshape for loss calculation
        # output: (B, max_len, vocab_size) -> (B * max_len, vocab_size)
        output_dim = output.shape[-1]
        output = output.reshape(-1, output_dim)
        
        # Target is shifted by 1 (we predict the next token)
        # target: (B, max_len) -> (B * max_len)
        target = formulas[:, 1:].reshape(-1)
        
        loss = criterion(output, target)
        loss.backward()
        
        # Gradient clipping for LSTM
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        total_loss += loss.item()
        
    return total_loss / len(dataloader)

def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for images, formulas in tqdm(dataloader, desc="Evaluating", leave=False):
            images = images.to(device)
            formulas = formulas.to(device)
            
            output = model(images, formulas)
            
            output_dim = output.shape[-1]
            output = output.reshape(-1, output_dim)
            target = formulas[:, 1:].reshape(-1)
            
            loss = criterion(output, target)
            total_loss += loss.item()
            
    return total_loss / len(dataloader)

if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    # 1. Build dataset
    # (In build_dataset, you might want to remove max_samples limit to train on full data)
    train_loader, val_loader, vocab = build_dataset("data")
    
    # 2. Initialize model
    model = Im2LatexModel(vocab_size=len(vocab)).to(device)
    
    # 3. Setup loss and optimizer
    # ignore_index=0 to not penalize PAD tokens
    criterion = nn.CrossEntropyLoss(ignore_index=0) 
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    
    # 4. Training loop
    epochs = 5
    for epoch in range(epochs):
        print(f"Epoch {epoch+1}/{epochs}")
        train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss = evaluate(model, val_loader, criterion, device)
        print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")


Using device: cpu
Vocabulary size: 583
Loaded 10 samples (dropped 0 missing images)
Loaded 1 samples (dropped 0 missing images)
Epoch 1/5


Training:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

Train Loss: 6.3699 | Val Loss: 6.1485
Epoch 2/5


Training:   0%|          | 0/1 [00:00<?, ?it/s]

...

Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

Train Loss: 4.4680 | Val Loss: 4.5783
Epoch 5/5


Training:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

Train Loss: 4.0235 | Val Loss: 4.2794


In [16]:
def generate_formula(model, image, vocab, max_len=150, device="cpu"):
    '''
    Greedy decoding for inference.
    '''
    model.eval()
    with torch.no_grad():
        image = image.to(device)  # (1, 1, H, W)
        
        # 1. Encode image
        encoder_out = model.encoder(image)  # (1, 64, 512)
        
        # 2. Initialize hidden state
        h, c = model.decoder.init_hidden(encoder_out)
        
        # 3. Start with <SOS> token
        current_token = torch.tensor([[vocab.token2idx[vocab.SOS]]], dtype=torch.long).to(device)
        
        generated_tokens = []
        
        for _ in range(max_len):
            # Embed current token
            embedded = model.decoder.embedding(current_token)  # (1, 1, embed_dim)
            
            # Get attention context
            context, _ = model.decoder.attention(encoder_out, h)  # context: (1, 512)
            
            # LSTM step
            lstm_input = torch.cat([embedded[:, 0, :], context], dim=1)
            h, c = model.decoder.lstm_cell(lstm_input, (h, c))
            
            # Predict next token
            pred = model.decoder.fc(h)  # (1, vocab_size)
            next_token_idx = pred.argmax(dim=1).item()
            
            if next_token_idx == vocab.token2idx[vocab.EOS]:
                break
                
            generated_tokens.append(next_token_idx)
            
            # Update current token
            current_token = torch.tensor([[next_token_idx]], dtype=torch.long).to(device)
            
        return vocab.decode(generated_tokens)

# Test inference on a validation sample
if __name__ == "__main__":
    import matplotlib.pyplot as plt
    
    images, formulas = next(iter(val_loader))
    sample_img = images[0:1] # shape (1, 1, H, W)
    target_formula = vocab.decode(formulas[0].tolist())
    
    pred_formula = generate_formula(model, sample_img, vocab, device=device)
    
    # Display the image
    plt.imshow(sample_img.squeeze().cpu().numpy(), cmap='gray')
    plt.axis('off')
    plt.show()
    
    print("Target  :", target_formula)
    print("Predict :", pred_formula)


ModuleNotFoundError: No module named 'matplotlib'